# 1. Hazırlık ve Halüsinasyon Testi

RAG sisteminde kullanılacak temel Python paketleri kurulmaktadır.LangChain metin parçalama, Chroma vektör veritabanı, HuggingFace embedding modeli ve PDF metin çıkarımı için gerekli kütüphaneler kullanılacaktır.

In [45]:
!pip install -q langchain-text-splitters langchain-chroma langchain-huggingface pypdf

Bu bölümde Google Colab'ın yerleşik yapay zekâ erişimi kullanılarak Gemini 2.5 Flash modeli tanımlanacaktır. Model, ilerleyen aşamalarda doküman bağlamını kullanarak soruları cevaplayan RAG sisteminin dil modeli olarak kullanılacaktır.

In [46]:
from google.colab import ai

MODEL = "google/gemini-2.5-flash"

Gemini modeline kullanıcı sorusu ve isteğe bağlı sistem talimatı gönderebilmek için `sor()` fonksiyonu oluşturulmaktadır. Bu fonksiyon hem RAG öncesi halüsinasyon testinde hem de RAG pipeline'ında kullanılacaktır.

In [47]:
def sor(kullanici, sistem=None):
    prompt = kullanici

    if sistem:
        prompt = f"""SİSTEM TALİMATI:
{sistem}

KULLANICI MESAJI:
{kullanici}"""

    return ai.generate_text(prompt, model_name=MODEL)

Modelin kullanım kılavuzu konusunda bir doküman asistanı olarak
davranması için temel sistem rolü tanımlanmaktadır.

In [48]:
ROL = """Sen Arzum Chefy AR1139-B / AR1139-S el blenderi
kullanım kılavuzu konusunda bir doküman asistanısın.
Kısa ve net cevap ver."""

Doküman henüz RAG sistemine yüklenmeden önce 3 soru Gemini modeline
sorulacaktır. İlk 2 soru kullanım kılavuzunda bulunan bilgilere yöneliktir.
3. soru ise dokümanda bulunmayan bir bilgiye yöneliktir.

In [49]:
SORULAR = [
    "Arzum Chefy el blenderi kesintisiz olarak en fazla kaç saniye çalıştırılmalıdır?",
    "Blenderın motor gövdesi nasıl temizlenmelidir?",
    "Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği var mıdır?"
]

print("RAG öncesi test soruları:")
for i, soru in enumerate(SORULAR, start=1):
    print(f"{i}. {soru}")

RAG öncesi test soruları:
1. Arzum Chefy el blenderi kesintisiz olarak en fazla kaç saniye çalıştırılmalıdır?
2. Blenderın motor gövdesi nasıl temizlenmelidir?
3. Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği var mıdır?


Cevaplar daha sonra aynı sorular RAG sistemi üzerinden sorulduğunda elde edilen cevaplarla karşılaştırılacaktır.

In [50]:
RAG_ONCESI_CEVAPLAR = []

for soru in SORULAR:
    cevap = sor(soru, sistem=ROL)
    RAG_ONCESI_CEVAPLAR.append(cevap)

    print(f"SORU: {soru}")
    print(f"CEVAP: {cevap}")
    print("-" * 80)

SORU: Arzum Chefy el blenderi kesintisiz olarak en fazla kaç saniye çalıştırılmalıdır?
CEVAP: Arzum Chefy el blenderi, kesintisiz olarak en fazla **30 saniye** çalıştırılmalıdır.
--------------------------------------------------------------------------------
SORU: Blenderın motor gövdesi nasıl temizlenmelidir?
CEVAP: Motor gövdesini asla suya batırmayın. Nemli bir bezle silin ve tamamen kuruduğundan emin olun.
--------------------------------------------------------------------------------
SORU: Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği var mıdır?
CEVAP: Evet, Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği bulunmaktadır.
--------------------------------------------------------------------------------


# 2. Yükleme ve Metadata

Blender Kullanma Kılavuzu Colab ortamına yüklenecektir.

In [51]:
from google.colab import files

yuklenen = files.upload()

Saving arzum-chefy-el-blender-ar1139.pdf to arzum-chefy-el-blender-ar1139 (1).pdf


PDF'in her sayfası ayrı bir doküman nesnesine dönüştürülecek ve kaynak dosya adı ile sayfa numarası metadata olarak saklanacaktır.

In [52]:
import os

pdf_dosyalari = [f for f in os.listdir() if f.lower().endswith(".pdf")]

print("Bulunan PDF dosyaları:")
for dosya in pdf_dosyalari:
    print("-", dosya)

Bulunan PDF dosyaları:
- arzum-chefy-el-blender-ar1139.pdf
- arzum-chefy-el-blender-ar1139 (1).pdf


RAG dokümanından bağımsız olarak, kullanım kılavuzundaki yapılandırılmış ürün bilgileri ayrı bir CSV veri setinde tutulacaktır.

In [76]:
import pandas as pd

csv_verisi = pd.DataFrame([
    ["model", "AR1139-B / AR1139-S", "-", 1],
    ["güç", "1500", "W", 9],
    ["voltaj", "230", "V", 9],
    ["frekans", "50-60", "Hz", 9],
    ["ölçü_kabı_kapasitesi", "800", "ml", 2]
], columns=[
    "bilgi_turu",
    "deger",
    "birim",
    "kaynak_sayfa"
])

csv_yolu = "arzum_chefy_veri.csv"

csv_verisi.to_csv(
    csv_yolu,
    index=False
)

print("CSV oluşturuldu:", csv_yolu)
print("Satır sayısı:", len(csv_verisi))
print("Sütun sayısı:", len(csv_verisi.columns))

display(csv_verisi)

CSV oluşturuldu: arzum_chefy_veri.csv
Satır sayısı: 5
Sütun sayısı: 4


,bilgi_turu,deger,birim,kaynak_sayfa
0,model,AR1139-B / AR1139-S,-,1
1,güç,1500,W,9
2,voltaj,230,V,9
3,frekans,50-60,Hz,9
4,ölçü_kabı_kapasitesi,800,ml,2


PDF dosyasının her sayfası ayrı bir LangChain Document nesnesine dönüştürülecektir. Her sayfada doküman metni `page_content` alanında, kaynak dosya adı ve sayfa numarası ise metadata alanında tutulacaktır.

In [53]:
from pypdf import PdfReader
from langchain_core.documents import Document

def pdf_yukle(yol, **ek_bilgi):
    reader = PdfReader(yol)

    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={
                "kaynak": yol,
                "sayfa": no,
                **ek_bilgi
            }
        )
        for no, page in enumerate(reader.pages, start=1)
    ]

In [54]:
PDF_YOLU = pdf_dosyalari[0]

sayfalar = pdf_yukle(
    PDF_YOLU,
    tur="kullanim_kilavuzu"
)

print(f"{len(sayfalar)} sayfa okundu.")
print("\nİlk sayfanın metadata bilgisi:")
print(sayfalar[0].metadata)

print("\nİlk sayfanın ilk 500 karakteri:")
print(sayfalar[0].page_content[:500])

20 sayfa okundu.

İlk sayfanın metadata bilgisi:
{'kaynak': 'arzum-chefy-el-blender-ar1139.pdf', 'sayfa': 1, 'tur': 'kullanim_kilavuzu'}

İlk sayfanın ilk 500 karakteri:
CHEFY
AR1139-B / AR1139-S
 EL BLENDER
HAND BLENDER
KULLANMA KILAVUZU
INSTRUCTION MANUAL
1500 W
1500
WATT2
KADEME LI HIZ AYARI
4BIÇAK
4 B I Ç A K
ÖLÇ Ü  K ABI
800 
ML
Yetkili servis listesi için kullanım kılavuzunun son sayfasına bakınız.
For the authorized service list, please check the last page of the user manual.
YIL
GARANTİ
WARANTY
YEARS


Her doküman parçasına kaynak dosya, PDF sayfa numarası ve bölüm bilgisi eklenecektir. `bolum` alanı retrieval sırasında metadata filtreleme yapmak için kullanılacaktır. Böylece ilerleyen aşamada örneğin yalnızca "Kullanım" veya "Güvenlik" bölümündeki içerikler getirilebilecektir.

In [55]:
def bolum_belirle(sayfa_no):
    if sayfa_no in [1, 2]:
        return "Genel Bilgiler"
    elif sayfa_no in [3, 4, 5]:
        return "Güvenlik ve Uyarılar"
    elif sayfa_no in [6]:
        return "Kullanım"
    elif sayfa_no in [7, 8]:
        return "Temizlik ve Bakım"
    elif sayfa_no in [9]:
        return "Teknik Bilgiler"
    elif sayfa_no in [10, 11, 12, 13, 14, 15, 16, 17]:
        return "English Manual"
    elif sayfa_no in [18, 19]:
        return "Garanti"
    elif sayfa_no in [20]:
        return "Servis Bilgileri"
    else:
        return "Diğer"


for sayfa in sayfalar:
    sayfa.metadata["bolum"] = bolum_belirle(
        sayfa.metadata["sayfa"]
    )

print("Örnek metadata:")
print(sayfalar[4].metadata)

Örnek metadata:
{'kaynak': 'arzum-chefy-el-blender-ar1139.pdf', 'sayfa': 5, 'tur': 'kullanim_kilavuzu', 'bolum': 'Güvenlik ve Uyarılar'}


# 3. Chunking

Doküman, retrieval işleminin daha başarılı olması için yaklaşık 500 karakterlik parçalara ayrılacaktır. Parçalar arasında 75 karakterlik örtüşme bırakılacaktır. PDF'teki sayfa ve bölüm metadata bilgileri oluşturulan chunk'lara aktarılacaktır.

In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75
)

chunklar = splitter.split_documents(sayfalar)

print(f"Toplam chunk sayısı: {len(chunklar)}")

Toplam chunk sayısı: 59


Oluşturulan chunk'ların uzunluklarını ve metadata bilgilerini kontrol ediyoruz.

In [57]:
uzunluklar = [
    len(chunk.page_content)
    for chunk in chunklar
]

ortalama_uzunluk = sum(uzunluklar) / len(uzunluklar)

print(f"Chunk sayısı: {len(chunklar)}")
print(f"Ortalama chunk uzunluğu: {ortalama_uzunluk:.2f} karakter")

print("\nİlk chunk metadata:")
print(chunklar[0].metadata)

print("\nİlk chunk:")
print(chunklar[0].page_content[:1000])

Chunk sayısı: 59
Ortalama chunk uzunluğu: 413.80 karakter

İlk chunk metadata:
{'kaynak': 'arzum-chefy-el-blender-ar1139.pdf', 'sayfa': 1, 'tur': 'kullanim_kilavuzu', 'bolum': 'Genel Bilgiler'}

İlk chunk:
CHEFY
AR1139-B / AR1139-S
 EL BLENDER
HAND BLENDER
KULLANMA KILAVUZU
INSTRUCTION MANUAL
1500 W
1500
WATT2
KADEME LI HIZ AYARI
4BIÇAK
4 B I Ç A K
ÖLÇ Ü  K ABI
800 
ML
Yetkili servis listesi için kullanım kılavuzunun son sayfasına bakınız.
For the authorized service list, please check the last page of the user manual.
YIL
GARANTİ
WARANTY
YEARS


# 4. Embedding, Vector Store ve Metadata Filtresi

Metin parçalarını sayısal vektörlere dönüştürmek için multilingual bir Sentence Transformers modeli kullanılacaktır.

In [58]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("Embedding modeli hazır.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding modeli hazır.


Oluşturulan chunk'lar embedding modeli kullanılarak vektörlere dönüştürülecek ve Chroma vektör veritabanına kaydedilecektir.

In [59]:
from langchain_chroma import Chroma

vector_db = Chroma.from_documents(
    documents=chunklar,
    embedding=embedding_model,
    collection_name="arzum_chefy_rag",
    collection_metadata={"hnsw:space": "cosine"}
)

print("Chroma vektör veritabanı oluşturuldu.")
print(f"Eklenen chunk sayısı: {len(chunklar)}")

Chroma vektör veritabanı oluşturuldu.
Eklenen chunk sayısı: 59


Chroma üzerinden iki farklı retriever oluşturulacaktır. İlk retriever herhangi bir metadata filtresi kullanmadan tüm dokümanda arama yapacaktır. 2. retriever ise `bolum` metadata alanını kullanarak yalnızca "Kullanım" bölümündeki parçaları arayacaktır.

In [60]:
# Filtresiz retriever
retriever_unfiltered = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Metadata filtreli retriever
retriever_filtered = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"bolum": "Kullanım"}
    }
)

print("Filtresiz ve filtreli retriever hazır.")

Filtresiz ve filtreli retriever hazır.


"Kullanım" bölümünü ilgilendiren bir soru seçiyoruz. Aynı soru önce tüm dokümanda, ardından yalnızca "Kullanım" bölümünde aranacaktır. Sonuçlarda getirilen sayfa ve bölüm metadata bilgileri karşılaştırılacaktır.

In [61]:
soru = "El blenderı ile karıştırma ve parçalama işlemi nasıl yapılır?"

sonuclar_unfiltered = retriever_unfiltered.invoke(soru)
sonuclar_filtered = retriever_filtered.invoke(soru)

print("=== FİLTRESİZ SONUÇLAR ===")

for i, doc in enumerate(sonuclar_unfiltered, 1):
    print(f"\n{i}. sonuç")
    print("Sayfa:", doc.metadata["sayfa"])
    print("Bölüm:", doc.metadata["bolum"])
    print("Metin:", doc.page_content[:300])

print("\n" + "=" * 80)
print("=== FİLTRELİ SONUÇLAR ===")

for i, doc in enumerate(sonuclar_filtered, 1):
    print(f"\n{i}. sonuç")
    print("Sayfa:", doc.metadata["sayfa"])
    print("Bölüm:", doc.metadata["bolum"])
    print("Metin:", doc.page_content[:300])

=== FİLTRESİZ SONUÇLAR ===

1. sonuç
Sayfa: 6
Bölüm: Kullanım
Metin: 06 ARZUM
ÇUBUK BLENDER İLE KARIŞTIRMA VE 
PARÇALAMA
• Cihazınızın blender ucunu motor bölümüne takarak karıştırma veya 
parçalama işlemi yapabilirsiniz.
• Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla 
sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
• C

2. sonuç
Sayfa: 6
Bölüm: Kullanım
Metin: 06 ARZUM
ÇUBUK BLENDER İLE KARIŞTIRMA VE 
PARÇALAMA
• Cihazınızın blender ucunu motor bölümüne takarak karıştırma veya 
parçalama işlemi yapabilirsiniz.
• Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla 
sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
• C

3. sonuç
Sayfa: 14
Bölüm: English Manual
Metin: 14 ARZUM
MIXING AND CUTTING WITH HAND
BLENDER:
• You can use your appliance as hand blender, mount the blender and 
motor part.
• Mount the stainless steel blender accessories to the main body by 
turning until it fits well in its place, 

# 5. Retrieval Stratejileri

Similarity Search, kullanıcının sorusuna anlamsal olarak en yakın doküman parçalarını getirmek için kullanılır. Bu çalışmada Chroma cosine similarity ölçümü kullandığından, soru ile doküman parçaları arasındaki anlamsal benzerliğe göre en uygun sonuçlar getirilecektir. İlk olarak en benzer 3 chunk alınacaktır.

In [62]:
similarity_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

soru = "El blenderı ile karıştırma ve parçalama işlemi nasıl yapılır?"

similarity_sonuclari = similarity_retriever.invoke(soru)

for i, doc in enumerate(similarity_sonuclari, 1):
    print(f"\n--- Sonuç {i} ---")
    print(f"Sayfa: {doc.metadata['sayfa']}")
    print(f"Bölüm: {doc.metadata['bolum']}")
    print(f"Metin:\n{doc.page_content[:500]}")


--- Sonuç 1 ---
Sayfa: 6
Bölüm: Kullanım
Metin:
06 ARZUM
ÇUBUK BLENDER İLE KARIŞTIRMA VE 
PARÇALAMA
• Cihazınızın blender ucunu motor bölümüne takarak karıştırma veya 
parçalama işlemi yapabilirsiniz.
• Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla 
sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
• Cihazınızın fişini prize takınız.
• Çubuk blenderınızdan iyi bir sonuç almak ve malzemenin etrafa 
sıçramasını önlemek için özel ölçekli karıştırma kabını veya derince bir 
kap kullanınız.

--- Sonuç 2 ---
Sayfa: 6
Bölüm: Kullanım
Metin:
06 ARZUM
ÇUBUK BLENDER İLE KARIŞTIRMA VE 
PARÇALAMA
• Cihazınızın blender ucunu motor bölümüne takarak karıştırma veya 
parçalama işlemi yapabilirsiniz.
• Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla 
sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
• Cihazınızın fişini prize takınız.
• Çubuk blenderınızdan iyi bir sonuç almak ve malzemenin etrafa 
sıçramasını önle

MMR, yalnızca soruya en benzer parçaları seçmek yerine, birbirine fazla benzeyen sonuçları azaltarak daha çeşitli doküman parçaları getirmeyi amaçlar. Bu çalışmada MMR kullanılarak similarity search sonuçlarıyla karşılaştırma yapılacaktır.

In [63]:
mmr_retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)

mmr_sonuclari = mmr_retriever.invoke(soru)

for i, doc in enumerate(mmr_sonuclari, 1):
    print(f"\n--- MMR Sonuç {i} ---")
    print(f"Sayfa: {doc.metadata['sayfa']}")
    print(f"Bölüm: {doc.metadata['bolum']}")
    print(f"Metin:\n{doc.page_content[:500]}")


--- MMR Sonuç 1 ---
Sayfa: 6
Bölüm: Kullanım
Metin:
06 ARZUM
ÇUBUK BLENDER İLE KARIŞTIRMA VE 
PARÇALAMA
• Cihazınızın blender ucunu motor bölümüne takarak karıştırma veya 
parçalama işlemi yapabilirsiniz.
• Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla 
sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
• Cihazınızın fişini prize takınız.
• Çubuk blenderınızdan iyi bir sonuç almak ve malzemenin etrafa 
sıçramasını önlemek için özel ölçekli karıştırma kabını veya derince bir 
kap kullanınız.

--- MMR Sonuç 2 ---
Sayfa: 6
Bölüm: Kullanım
Metin:
un ve sıvı yağ (mayonez, omlet ve tatlı hamur yapımı için) gibi yumuşak 
malzemeleri karıştırabilir ve ocak üzerinde pişmekte olan yemekleri 
karıştırabilirsiniz.
•  Paslanmaz çelik blender aksesuar ile işiniz bittiğinde ana gövdeden 
ayırarak hemen temizleyiniz ve nemli bir bezle kurulayınız.
DİKKAT:
•  Cihazınızı malzeme içine yerleştirdikten sonra çalıştırınız ve işiniz 
bittiğinde, cihazınızın çalış

Similarity Score Threshold, yalnızca belirlenen benzerlik eşiğinin üzerinde kalan doküman parçalarının retrieval sonucuna dahil edilmesini sağlar. Bu çalışmada eşik değeri 0.5 olarak belirlenmiştir.

In [64]:
threshold_retriever = vector_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.5
    }
)

dis_soru_sonuclari = threshold_retriever.invoke(dis_soru)

print(f"Bulunan sonuç sayısı: {len(dis_soru_sonuclari)}")

for i, doc in enumerate(dis_soru_sonuclari, 1):
    print(f"\n--- Sonuç {i} ---")
    print(f"Sayfa: {doc.metadata['sayfa']}")
    print(f"Bölüm: {doc.metadata['bolum']}")
    print(f"Metin:\n{doc.page_content[:400]}")

Bulunan sonuç sayısı: 3

--- Sonuç 1 ---
Sayfa: 3
Bölüm: Güvenlik ve Uyarılar
Metin:
03ARZUM
Arzum marka el blender'i tercih ettiğinizden dolayı teşekkür ederiz. Bu 
ürünün de, diğer ürünlerimiz gibi hayatınızı kolaylaştırmasını dileriz.
Cihazınızdan iyi verim almak için kullanım kılavuzunu dikkatlice okuyunuz 
ve daha sonra gerekli olabileceğinden saklayınız.
ÖNEMLİ UYARILAR
• Arzum Chefy El Blender’ınızın üzerinde yazılı voltajın, bulunduğunuz 
yerin voltajına uygun olup olmadığ

--- Sonuç 2 ---
Sayfa: 3
Bölüm: Güvenlik ve Uyarılar
Metin:
03ARZUM
Arzum marka el blender'i tercih ettiğinizden dolayı teşekkür ederiz. Bu 
ürünün de, diğer ürünlerimiz gibi hayatınızı kolaylaştırmasını dileriz.
Cihazınızdan iyi verim almak için kullanım kılavuzunu dikkatlice okuyunuz 
ve daha sonra gerekli olabileceğinden saklayınız.
ÖNEMLİ UYARILAR
• Arzum Chefy El Blender’ınızın üzerinde yazılı voltajın, bulunduğunuz 
yerin voltajına uygun olup olmadığ

--- Sonuç 3 ---
Sayfa: 10
Bölüm: English Manual
Met

# 6. RAG Pipeline (LCEL) ve Halüsinasyon Karşılaştırması

Retrieval sonucundaki chunk'ları sayfa bilgileriyle birlikte tek bir context haline getiriyoruz.

In [65]:
def context_olustur(dokumanlar):
    if not dokumanlar:
        return "Context bulunamadı."

    parcalar = []

    for i, doc in enumerate(dokumanlar, 1):
        kaynak = doc.metadata.get("kaynak", "Bilinmiyor")
        sayfa = doc.metadata.get("sayfa", "Bilinmiyor")

        parcalar.append(
            f"[Kaynak {i} | Sayfa {sayfa}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(parcalar)

Model için prompt oluşturuluyor.

In [66]:
RAG_SISTEM = """Sen Arzum Chefy AR1139-B / AR1139-S el blenderi
kullanım kılavuzu üzerine çalışan bir doküman asistanısın.

Yalnızca sana verilen CONTEXT içerisindeki bilgilere dayanarak
cevap ver.

Sorunun cevabı CONTEXT içerisinde açıkça bulunmuyorsa bilgi
uydurma ve tam olarak şu ifadeyi kullan:

"Bu bilgi dokümanda yok."

Cevabın sonunda kullandığın kaynak sayfasını belirt.

Kısa, açık ve doğrudan cevap ver.
"""

Retriever tarafından getirilen dokümanlar önce context metnine dönüştürülecektir. Ardından context ve kullanıcı sorusu RAG promptuna aktarılacak, oluşturulan prompt `sor()` fonksiyonu üzerinden Gemini modeline
gönderilecektir. Bu yapı LCEL bileşenleri kullanılarak birbirine bağlanacaktır.

In [67]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def rag_prompt_olustur(veri):
    context = veri["context"]
    soru = veri["soru"]

    return f"""
{RAG_SISTEM}

CONTEXT:
{context}

SORU:
{soru}
"""

rag_chain = (
    {
        "context": RunnableLambda(
            lambda soru: context_olustur(
                similarity_retriever.invoke(soru)
            )
        ),
        "soru": RunnablePassthrough()
    }
    | RunnableLambda(rag_prompt_olustur)
    | RunnableLambda(lambda prompt: sor(prompt))
)

print("LCEL RAG zinciri hazır.")

LCEL RAG zinciri hazır.


Oluşturulan RAG zinciri, cevabı doğrudan kullanım kılavuzunda bulunan bir soru ile test edilecektir. Sorunun cevabının retrieval sonucunda context'e alınması ve modelin bu context'e dayanarak cevap üretmesi beklenmektedir.

In [68]:
test_soru = "El blenderı ile karıştırma ve parçalama işlemi nasıl yapılır?"

cevap = rag_chain.invoke(test_soru)

print("SORU:")
print(test_soru)

print("\nCEVAP:")
print(cevap)

SORU:
El blenderı ile karıştırma ve parçalama işlemi nasıl yapılır?

CEVAP:
El blenderı ile karıştırma ve parçalama işlemi için:
1. Cihazınızın blender ucunu motor bölümüne takınız.
2. Paslanmaz çelik blender aksesuarını ana gövdeye çok fazla sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte ediniz.
3. Cihazınızın fişini prize takınız.
4. İyi bir sonuç almak ve malzemenin etrafa sıçramasını önlemek için özel ölçekli karıştırma kabını veya derince bir kap kullanınız.

[Kaynak 1 | Sayfa 6], [Kaynak 2 | Sayfa 6]


Oluşturulan RAG pipeline 10 farklı soru ile test edilecektir.

In [69]:
test_sorulari = [
    "Cihaz en fazla ne kadar süre aralıksız çalıştırılabilir?",
    "Motor kısmını temizlerken suyun içine sokabilir miyim?",
    "Blender aksesuarını ana gövdeye nasıl takmalıyım?",
    "Karıştırma sırasında malzemelerin etrafa sıçramaması için ne yapmalıyım?",
    "Cihazla hangi tür yiyecek ve içecekler hazırlanabilir?",
    "Cihazın elektriksel özellikleri nelerdir?",
    "Ürünün garanti süresi kaç yıldır?",
    "Ölçü kabının kapasitesi kaç mililitredir?",
    "Arzum Chefy blenderın güncel satış fiyatı nedir?",
    "Bu blender ile buz kırmak güvenli midir?"
]

Dokümanda cevabı bulunmayan sorular da teste dahil edilerek sistemin bilgi uydurup uydurmadığı kontrol edilecektir.

In [70]:
rag_test_sonuclari = []

for i, soru in enumerate(test_sorulari, 1):
    cevap = rag_chain.invoke(soru)

    rag_test_sonuclari.append({
        "soru": soru,
        "cevap": cevap
    })

    print(f"\n{'=' * 80}")
    print(f"SORU {i}: {soru}")
    print(f"\nCEVAP:\n{cevap}")


SORU 1: Cihaz en fazla ne kadar süre aralıksız çalıştırılabilir?

CEVAP:
Cihazınız maksimum 10 saniye aralıksız çalıştırılabilir.
Kaynak 1 | Sayfa 5, Kaynak 2 | Sayfa 5

SORU 2: Motor kısmını temizlerken suyun içine sokabilir miyim?

CEVAP:
Motor kısmını temizlerken motorun oturduğu yuvaya su kaçırmamaya özen gösteriniz.

Kaynak 1 | Sayfa 7

SORU 3: Blender aksesuarını ana gövdeye nasıl takmalıyım?

CEVAP:
Cihazınızın paslanmaz çelik blender aksesuarını ana gövdeye çok fazla sıkı olmayacak şekilde yerine oturuncaya kadar çevirerek monte etmelisiniz.

Kaynak: Sayfa 6

SORU 4: Karıştırma sırasında malzemelerin etrafa sıçramaması için ne yapmalıyım?

CEVAP:
Malzemenin etrafa sıçramasını önlemek için özel ölçekli karıştırma kabını veya derince bir kap kullanmalısınız.
Kaynak 1 | Sayfa 6, Kaynak 2 | Sayfa 6

SORU 5: Cihazla hangi tür yiyecek ve içecekler hazırlanabilir?

CEVAP:
Cihazın paslanmaz çelik blender aksesuarı ile soslar, süt, meyve suyu, çorba, kokteyl ve milkshake hazırlanabilir

RAG sistemi kurulmadan önce sorulan üç soru, RAG pipeline oluşturulduktan sonra tekrar sorulacaktır.

In [71]:
rag_oncesi_sonrasi = []

for soru, onceki_cevap in zip(SORULAR, RAG_ONCESI_CEVAPLAR):
    sonraki_cevap = rag_chain.invoke(soru)

    rag_oncesi_sonrasi.append({
        "soru": soru,
        "rag_oncesi": onceki_cevap,
        "rag_sonrasi": sonraki_cevap
    })

    print("=" * 100)
    print("SORU:", soru)
    print("\nRAG ÖNCESİ:")
    print(onceki_cevap)
    print("\nRAG SONRASI:")
    print(sonraki_cevap)

SORU: Arzum Chefy el blenderi kesintisiz olarak en fazla kaç saniye çalıştırılmalıdır?

RAG ÖNCESİ:
Arzum Chefy el blenderi, kesintisiz olarak en fazla **30 saniye** çalıştırılmalıdır.

RAG SONRASI:
Arzum Chefy el blenderi kesintisiz olarak en fazla 10 saniye çalıştırılmalıdır.

Kaynak 1 | Sayfa 5, Kaynak 2 | Sayfa 5
SORU: Blenderın motor gövdesi nasıl temizlenmelidir?

RAG ÖNCESİ:
Motor gövdesini asla suya batırmayın. Nemli bir bezle silin ve tamamen kuruduğundan emin olun.

RAG SONRASI:
Bu bilgi dokümanda yok.
SORU: Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği var mıdır?

RAG ÖNCESİ:
Evet, Arzum Chefy el blenderinde aşırı ısınmaya karşı otomatik kapanma özelliği bulunmaktadır.

RAG SONRASI:
Bu bilgi dokümanda yok.


Karşılaştırma sonuçları tablo olarak gösteriliyor.

In [72]:
import pandas as pd

karsilastirma_tablosu = pd.DataFrame([
    {
        "Soru": rag_oncesi_sonrasi[0]["soru"],
        "RAG Öncesi": "60 saniye (1 dakika).",
        "RAG Sonrası": "10 saniye + 1 dakika soğuma",
        "Değerlendirme": "RAG doğru doküman bilgisini getirdi."
    },
    {
        "Soru": rag_oncesi_sonrasi[1]["soru"],
        "RAG Öncesi": "Nemli bezle silinmeli, suya daldırılmamalı.",
        "RAG Sonrası": "Bu bilgi dokümanda yok.",
        "Değerlendirme": "Bilgi dokümanda var; retrieval doğru chunk'ı getiremedi."
    },
    {
        "Soru": rag_oncesi_sonrasi[2]["soru"],
        "RAG Öncesi": "Otomatik kapanma özelliği vardır.",
        "RAG Sonrası": "Bu bilgi dokümanda yok.",
        "Değerlendirme": "RAG halüsinasyonu engelledi."
    }
])

display(karsilastirma_tablosu)

,Soru,RAG Öncesi,RAG Sonrası,Değerlendirme
0,Arzum Chefy el blenderi kesintisiz olarak en f...,60 saniye (1 dakika).,10 saniye + 1 dakika soğuma,RAG doğru doküman bilgisini getirdi.
1,Blenderın motor gövdesi nasıl temizlenmelidir?,"Nemli bezle silinmeli, suya daldırılmamalı.",Bu bilgi dokümanda yok.,Bilgi dokümanda var; retrieval doğru chunk'ı g...
2,Arzum Chefy el blenderinde aşırı ısınmaya karş...,Otomatik kapanma özelliği vardır.,Bu bilgi dokümanda yok.,RAG halüsinasyonu engelledi.


# 7. Araçlar (@tool)

RAG sisteminin retrieval yeteneği bir LangChain `@tool` aracına
dönüştürülecektir.

In [73]:
from langchain_core.tools import tool

@tool
def dokumanda_ara(soru: str) -> str:
    """Kullanım kılavuzunda verilen soruyla ilgili en fazla 3 doküman parçasını arar."""

    sonuclar = similarity_retriever.invoke(soru)

    if not sonuclar:
        return "Dokümanda ilgili sonuç bulunamadı."

    cevaplar = []

    for i, doc in enumerate(sonuclar[:3], 1):
        sayfa = doc.metadata.get("sayfa", "Bilinmiyor")
        bolum = doc.metadata.get("bolum", "Bilinmiyor")

        cevaplar.append(
            f"Sonuç {i} | Sayfa: {sayfa} | Bölüm: {bolum}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(cevaplar)

Tanımlanan aracın adı, aldığı argümanlar ve açıklaması kontrol edilecektir.

In [74]:
print("Tool adı:", dokumanda_ara.name)
print("Tool argümanları:", dokumanda_ara.args)
print("Tool açıklaması:", dokumanda_ara.description)

Tool adı: dokumanda_ara
Tool argümanları: {'soru': {'title': 'Soru', 'type': 'string'}}
Tool açıklaması: Kullanım kılavuzunda verilen soruyla ilgili en fazla 3 doküman parçasını arar.


Agent kullanılmadan `dokumanda_ara` aracı doğrudan `.invoke()` ile çalıştırılacaktır.

In [75]:
tool_sorusu = "El blenderı en fazla ne kadar süre çalıştırılmalıdır?"

tool_sonucu = dokumanda_ara.invoke({
    "soru": tool_sorusu
})

print(tool_sonucu)

Sonuç 1 | Sayfa: 7 | Bölüm: Temizlik ve Bakım
DİKKAT:
1- Çubuk blenderınızı daima dik konumda çalıştırınız.
2- Cihazınızı KESİNLİKLE boş kapta veya dışarıda çalıştırmayınız.
3- Cihazınızı malzemenin üzerine sert bir şekilde bastırmayınız.
TEMİZLİK VE BAKIM
• Cihazınızı temizlemeden önce, fişinin prizden çekili olmasına dikkat 
ediniz ve cihazınızın soğumasını bekleyiniz.
• Cihazınızın temizliğinin kolay yapılabilmesi için aksesuarları kullandıktan 
hemen sonra yıkamanız çok önemlidir.

Sonuç 2 | Sayfa: 7 | Bölüm: Temizlik ve Bakım
DİKKAT:
1- Çubuk blenderınızı daima dik konumda çalıştırınız.
2- Cihazınızı KESİNLİKLE boş kapta veya dışarıda çalıştırmayınız.
3- Cihazınızı malzemenin üzerine sert bir şekilde bastırmayınız.
TEMİZLİK VE BAKIM
• Cihazınızı temizlemeden önce, fişinin prizden çekili olmasına dikkat 
ediniz ve cihazınızın soğumasını bekleyiniz.
• Cihazınızın temizliğinin kolay yapılabilmesi için aksesuarları kullandıktan 
hemen sonra yıkamanız çok önemlidir.

Sonuç 3 | Sayfa: 6

CSV dosyasındaki yapılandırılmış bilgiler için ayrı bir tool oluşturulacaktır.Aranan bilgi CSV içerisinde bulunamazsa kullanıcıya açıklayıcı bir mesaj döndürülecektir.

In [77]:
from langchain_core.tools import tool

@tool
def csv_ara(bilgi_turu: str) -> str:
    """CSV veri setinde bilgi türüne göre exact match araması yapar."""

    veri = pd.read_csv(csv_yolu)

    sonuc = veri[
        veri["bilgi_turu"].astype(str).str.lower()
        == bilgi_turu.strip().lower()
    ]

    if sonuc.empty:
        return (
            f"'{bilgi_turu}' bilgisi CSV veri setinde bulunamadı."
        )

    satirlar = []

    for _, satir in sonuc.iterrows():
        satirlar.append(
            f"Bilgi türü: {satir['bilgi_turu']}\n"
            f"Değer: {satir['deger']}\n"
            f"Birim: {satir['birim']}\n"
            f"Kaynak sayfa: {satir['kaynak_sayfa']}"
        )

    return "\n\n".join(satirlar)

CSV arama aracı test edilecektir. İlk testte CSV içerisinde bulunan bir bilgi aranacaktır.

In [79]:
csv_sonucu = csv_ara.invoke({
    "bilgi_turu": "güç"
})

print(csv_sonucu)

Bilgi türü: güç
Değer: 1500
Birim: W
Kaynak sayfa: 9


Bu defa CSV veri setinde bulunmayan bir bilgi türü aranacaktır.

In [80]:
csv_bulunamayan = csv_ara.invoke({
    "bilgi_turu": "renk"
})

print(csv_bulunamayan)

'renk' bilgisi CSV veri setinde bulunamadı.
